# LM Studio Financial Sentiment Analysis

This notebook performs financial sentiment analysis (Positive / Negative / Neutral) using a local LM Studio server instead of a fine-tuned FinBERT model.

**Setup requirements:**
- LM Studio running locally with Qwen 35B loaded
- Local server enabled on `http://127.0.0.1:1234`
- `openai` Python package installed (`pip install openai`)

In [ ]:
import json
from openai import OpenAI
print("Imports OK")

In [ ]:
client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio",  # required by the SDK; LM Studio ignores the value
)

# Check available models and confirm the server is reachable
try:
    models = client.models.list()
    model_ids = [m.id for m in models.data]
    print("Server reachable. Available models:")
    for mid in model_ids:
        print(" -", mid)
except Exception as e:
    print(f"ERROR: Could not reach LM Studio server — {e}")
    print("Make sure LM Studio is open, a model is loaded, and the local server is started on port 1234.")

In [ ]:
# Paste the model ID printed above (e.g. "qwen2.5-32b-instruct")
MODEL = "qwen-35b"  # <-- update this to match the exact ID shown above

In [ ]:
import re

def get_local_sentiment(text: str) -> dict:
    """
    Queries the local LM Studio server and returns:
      {"sentiment": "Positive" | "Negative" | "Neutral", "confidence_score": float}
    """
    system_prompt = (
        "You are a financial expert specializing in market sentiment analysis. "
        "Analyze the sentiment of the provided financial text and respond ONLY with "
        "a valid JSON object in this exact format: "
        '{"sentiment": "Positive" | "Negative" | "Neutral", "confidence_score": <float between 0 and 1>}. '
        "Do not include any explanation, markdown, or extra text — just the raw JSON object."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analyze the sentiment of this financial text:\n\n{text}"},
        ],
        temperature=0.1,
    )
    raw = response.choices[0].message.content
    # Extract the first {...} block in case the model adds surrounding text
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in model response:\n{raw}")
    return json.loads(match.group())

print("get_local_sentiment() defined")

## Smoke Test

In [ ]:
result = get_local_sentiment("Earnings beat expectations; shares rallied 5%.")
print(result)
# Expected: {"sentiment": "Positive", "confidence_score": ~0.95}

## Demo Examples

The following two examples mirror those used in `finbert_training.ipynb` so results can be compared directly against the FinBERT baseline (Apple → Negative, Naspers → Positive).

In [ ]:
# Demo 1 — Apple earnings warning (FinBERT baseline: Negative, avg sentiment score -0.91)
text1 = (
    "Later that day Apple said it was revising down its earnings expectations in "
    "the fourth quarter of 2018, largely because of lower sales and signs of economic weakness in China. "
    "The news rapidly infected financial markets. Apple's share price fell by around 7% in after-hours "
    "trading and the decline was extended to more than 10% when the market opened. The dollar fell "
    "by 3.7% against the yen in a matter of minutes after the announcement, before rapidly recovering "
    "some ground. Asian stockmarkets closed down on January 3rd and European ones opened lower. "
    "Yields on government bonds fell as investors fled to the traditional haven in a market storm."
)

result1 = get_local_sentiment(text1)
print(result1)

In [ ]:
# Demo 2 — Naspers IPO (FinBERT baseline: Positive, avg sentiment score +0.51)
text2 = (
    "Shares in the spin-off of South African e-commerce group Naspers surged more than 25% "
    "in the first minutes of their market debut in Amsterdam on Wednesday. Bob van Dijk, CEO of "
    "Naspers and Prosus Group poses at Amsterdam's stock exchange, as Prosus begins trading on the "
    "Euronext stock exchange in Amsterdam, Netherlands, September 11, 2019. REUTERS/Piroschka van de Wouw "
    "Prosus comprises Naspers' global empire of consumer internet assets, with the jewel in the crown a "
    "31% stake in Chinese tech titan Tencent. There is 'way more demand than is even available, so that's "
    "good,' said the CEO of Euronext Amsterdam, Maurice van Tilburg. 'It's going to be an interesting "
    "hour of trade after opening this morning.' Euronext had given an indicative price of 58.70 euros "
    "per share for Prosus, implying a market value of 95.3 billion euros ($105 billion). The shares "
    "jumped to 76 euros on opening and were trading at 75 euros at 0719 GMT."
)

result2 = get_local_sentiment(text2)
print(result2)